# LeetCode 2124 — Check if All A's Appears Before All B's bằng Python

**Link đề:** https://leetcode.com/problems/check-if-all-as-appears-before-all-bs/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def checkString(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Dịch câu "**mọi** `a` đứng trước **mọi** `b`" thành điều kiện kiểm tra được, và hiểu cách kiểm tra **phủ định** ("có `b` nào đứng trước `a` không?").
2. Đi từ **thử mọi cặp** `O(n²)` → **vị trí `a` cuối < vị trí `b` đầu** → **cờ (flag) "đã gặp b"** một lượt → mẹo **`"ba" not in s`**.
3. Nắm cú pháp: `str.find`, `str.rfind`, `in` với chuỗi con, `all`/`any`, `zip(s, s[1:])`, `sorted(s)`, `"".join`.
4. Hiểu "**vacuous truth**" (đúng một cách hiển nhiên): chuỗi không có `a` (hoặc không có `b`) vẫn trả `True`.
5. Trace, chứng minh, Time/Space, test vét cạn, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

Cho chuỗi `s` chỉ gồm `'a'` và `'b'`. Trả `True` nếu **mọi** `'a'` xuất hiện **trước mọi** `'b'`; ngược lại `False`.

Nói cách khác: `s` phải có dạng `aaa...abbb...b` (một khối `a` rồi một khối `b`, mỗi khối có thể rỗng).

| `s` | Kết quả | Giải thích |
|---|:---:|---|
| `"aaabbb"` | `True` | a ở 0,1,2; b ở 3,4,5 |
| `"abab"` | `False` | có `a` ở chỉ số 2 nằm sau `b` ở chỉ số 1 |
| `"bbb"` | `True` | không có `a` nào → điều kiện "mọi a…" đúng hiển nhiên |

**Constraints:** `1 <= len(s) <= 100`, `s[i]` là `'a'` hoặc `'b'`.

In [1]:
examples = [("aaabbb", True), ("abab", False), ("bbb", True)]
for s, exp in examples:
    print(f"{s!r:9} -> {exp}")

'aaabbb'  -> True
'abab'    -> False
'bbb'     -> True


### "Vacuous truth" — vì sao `"bbb"` là `True`?

Mệnh đề "**mọi** phần tử của tập rỗng đều thỏa P" luôn **đúng**, vì không có phản ví dụ nào. Python cũng làm vậy: `all([])` là `True`.

Tương tự `"aaa"` (không có `b`) cũng `True`.

In [2]:
print(all([]))                       # True
print(any([]))                       # False
a_positions = []                     # "bbb" không có a
print(all(i < 3 for i in a_positions))

True
False
True


## 2. Nền tảng Python cần dùng

### 2.1. `enumerate(s)` lấy chỉ số từng ký tự

In [3]:
s = "abab"
print([i for i, ch in enumerate(s) if ch == "a"])   # vị trí các a
print([i for i, ch in enumerate(s) if ch == "b"])   # vị trí các b

[0, 2]
[1, 3]


### 2.2. `str.find(x)` và `str.rfind(x)`

- `s.find("b")`: chỉ số **đầu tiên** của `"b"`, không có → `-1`.
- `s.rfind("a")`: chỉ số **cuối cùng** của `"a"`, không có → `-1`.

In [4]:
for s in ["aaabbb", "abab", "bbb", "aaa"]:
    print(f"{s!r:9} find('b')={s.find('b'):>2}  rfind('a')={s.rfind('a'):>2}")

'aaabbb'  find('b')= 3  rfind('a')= 2
'abab'    find('b')= 1  rfind('a')= 2
'bbb'     find('b')= 0  rfind('a')=-1
'aaa'     find('b')=-1  rfind('a')= 2


### 2.3. `sub in s` — kiểm tra chuỗi con liên tiếp

In [5]:
print("ba" in "abab", "ba" in "aaabbb", "ba" in "bbb")

True False False


### 2.4. `zip(s, s[1:])` — duyệt từng cặp ký tự **kề nhau**

`s[1:]` là `s` bỏ ký tự đầu. `zip` ghép `s[0]` với `s[1]`, `s[1]` với `s[2]`, … và dừng khi chuỗi ngắn hơn hết.

In [6]:
s = "aabab"
print(list(zip(s, s[1:])))

[('a', 'a'), ('a', 'b'), ('b', 'a'), ('a', 'b')]


### 2.5. `sorted(s)` với chuỗi

`sorted("bab")` trả **list** ký tự đã sắp xếp `['a','b','b']`. Muốn có lại chuỗi: `"".join(sorted(s))`.

In [7]:
print(sorted("abab"), "".join(sorted("abab")))

['a', 'a', 'b', 'b'] aabb


### 2.6. Biến cờ (flag) kiểu `bool`

Một biến `True/False` ghi nhớ "đã từng xảy ra sự kiện X chưa". Bắt đầu `False`, bật `True` khi sự kiện xảy ra, không bao giờ tắt lại.

In [8]:
seen_b = False
for ch in "aab":
    if ch == "b":
        seen_b = True
    print(ch, "-> seen_b =", seen_b)

a -> seen_b = False
a -> seen_b = False
b -> seen_b = True


## 3. Cách 1 — Trực giác: kiểm tra đúng nghĩa đen bằng mọi cặp

"Mọi `a` đứng trước mọi `b`" ⇔ với **mọi** cặp `(i, j)` mà `s[i] == 'a'` và `s[j] == 'b'`, ta có `i < j`.

Kiểm tra **phủ định** dễ hơn (gợi ý 1 của đề): tìm một cặp **vi phạm** `s[i] == 'b'`, `s[j] == 'a'` với `i < j`. Có ⇒ `False`.

In [9]:
def check_string_pairs(s: str) -> bool:
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n):
            if s[i] == "b" and s[j] == "a":   # b đứng trước a -> vi phạm
                return False
    return True

for s, exp in examples:
    print(f"{s!r:9} -> {check_string_pairs(s)} (kỳ vọng {exp})")

'aaabbb'  -> True (kỳ vọng True)
'abab'    -> False (kỳ vọng False)
'bbb'     -> True (kỳ vọng True)


## 4. Nút thắt — `O(n²)` cặp, trong khi chỉ cần hai vị trí

Với `n = 100` có ~5000 cặp — vẫn nhanh, nhưng lãng phí. Quan sát:

- Vi phạm tồn tại ⇔ có một `b` đứng trước một `a`.
- Chọn `b` **sớm nhất** và `a` **muộn nhất** là "có lợi nhất" cho việc vi phạm.
- Vậy: vi phạm ⇔ `first_b < last_a`.

Chỉ cần **hai con số**: vị trí `b` đầu tiên và vị trí `a` cuối cùng!

In [10]:
for s in ["aaabbb", "abab", "bbb", "aaa", "ba"]:
    fb, la = s.find("b"), s.rfind("a")
    print(f"{s!r:9} first_b={fb:>2}, last_a={la:>2} -> vi phạm? {fb != -1 and la != -1 and fb < la}")

'aaabbb'  first_b= 3, last_a= 2 -> vi phạm? False
'abab'    first_b= 1, last_a= 2 -> vi phạm? True
'bbb'     first_b= 0, last_a=-1 -> vi phạm? False
'aaa'     first_b=-1, last_a= 2 -> vi phạm? False
'ba'      first_b= 0, last_a= 1 -> vi phạm? True


## 5. Bước đột phá — hai cách nhìn gọn hơn nữa

**(a) Một lượt duyệt với cờ:** đi từ trái sang phải; khi đã gặp `b` mà sau đó lại gặp `a` ⇒ vi phạm. Đây chính là "first_b < last_a" nhưng không cần tìm hai vị trí trước.

**(b) Chuỗi con `"ba"`** (gợi ý 2 của đề): nếu có `b` đứng trước `a` (không nhất thiết kề nhau), thì giữa chúng chắc chắn có một chỗ **chuyển** từ `b` sang `a` **kề nhau**. Lý do: đi từ vị trí `b` đó sang phải tới vị trí `a`, ký tự đổi từ `b` sang `a` ít nhất một lần, và lần đổi đầu tiên là một cặp kề `"ba"`.

```text
s = "a b b a b"
       ^ b ở 1, a ở 3: đi 1 -> 2 -> 3:  b, b, a   -> cặp kề (2,3) = "ba"
```

Vậy: `s` hợp lệ ⇔ `"ba" not in s`.

In [11]:
for s in ["aaabbb", "abab", "bbb", "abba", "aabbbbbba"]:
    print(f"{s!r:11} 'ba' in s? {'ba' in s} -> kết quả {'ba' not in s}")

'aaabbb'    'ba' in s? False -> kết quả True
'abab'      'ba' in s? True -> kết quả False
'bbb'       'ba' in s? False -> kết quả True
'abba'      'ba' in s? True -> kết quả False
'aabbbbbba' 'ba' in s? True -> kết quả False


## 6. Lời giải chính (một lượt với cờ) — mổ xẻ rồi ghép

### 6.1. Đoạn 1: bật cờ khi gặp `b`

In [12]:
s = "aabab"
seen_b = False
for i, ch in enumerate(s):
    if ch == "b":
        seen_b = True
    print(i, ch, seen_b)

0 a False
1 a False
2 b True
3 a True
4 b True


### 6.2. Đoạn 2: gặp `a` khi cờ đã bật ⇒ vi phạm, dừng ngay

In [13]:
def check_string(s: str) -> bool:
    seen_b = False
    for ch in s:
        if ch == "b":
            seen_b = True
        elif seen_b:          # ch == 'a' và đã thấy b trước đó
            return False
    return True

for s, exp in examples:
    print(f"{s!r:9} -> {check_string(s)} (kỳ vọng {exp})")

'aaabbb'  -> True (kỳ vọng True)
'abab'    -> False (kỳ vọng False)
'bbb'     -> True (kỳ vọng True)


### 6.3. Vì sao viết `elif seen_b` mà không cần `ch == "a"`?

Chuỗi chỉ có `'a'` và `'b'`. Nhánh `elif` chỉ chạy khi `ch != "b"`, tức `ch == "a"`. Nếu đề cho thêm ký tự khác, ta phải viết rõ `elif ch == "a" and seen_b`.

## 7. TRACE từng bước

### 7.1. Bảng trace — `s = "abab"`

| i | ch | seen_b trước | Hành động | seen_b sau |
|:---:|:---:|:---:|---|:---:|
| 0 | a | False | `a` khi chưa thấy `b` → OK | False |
| 1 | b | False | bật cờ | True |
| 2 | a | True | `a` sau `b` → **return False** | — |
| 3 | b | — | không chạy tới | — |

In [14]:
def trace_check_string(s):
    seen_b = False
    print(f"s = {s!r}")
    for i, ch in enumerate(s):
        before = seen_b
        if ch == "b":
            seen_b = True
            act = "gặp b -> bật cờ"
        elif seen_b:
            print(f"  i={i}: ch={ch!r}, seen_b={before} -> 'a' sau 'b' => return False")
            return False
        else:
            act = "gặp a khi chưa có b -> OK"
        print(f"  i={i}: ch={ch!r}, seen_b {before} -> {seen_b}: {act}")
    print("  hết chuỗi, không có vi phạm => return True")
    return True

trace_check_string("abab")

s = 'abab'
  i=0: ch='a', seen_b False -> False: gặp a khi chưa có b -> OK
  i=1: ch='b', seen_b False -> True: gặp b -> bật cờ
  i=2: ch='a', seen_b=True -> 'a' sau 'b' => return False


False

### 7.2. Trường hợp hợp lệ và trường hợp không có `a`

In [15]:
trace_check_string("aaabbb")
print()
trace_check_string("bbb")

s = 'aaabbb'
  i=0: ch='a', seen_b False -> False: gặp a khi chưa có b -> OK
  i=1: ch='a', seen_b False -> False: gặp a khi chưa có b -> OK
  i=2: ch='a', seen_b False -> False: gặp a khi chưa có b -> OK
  i=3: ch='b', seen_b False -> True: gặp b -> bật cờ
  i=4: ch='b', seen_b True -> True: gặp b -> bật cờ
  i=5: ch='b', seen_b True -> True: gặp b -> bật cờ
  hết chuỗi, không có vi phạm => return True

s = 'bbb'
  i=0: ch='b', seen_b False -> True: gặp b -> bật cờ
  i=1: ch='b', seen_b True -> True: gặp b -> bật cờ
  i=2: ch='b', seen_b True -> True: gặp b -> bật cờ
  hết chuỗi, không có vi phạm => return True


True

### 7.3. Trace cách "ba" bằng các cặp kề nhau

In [16]:
def trace_pairs(s):
    for i, (x, y) in enumerate(zip(s, s[1:])):
        print(f"  cặp ({i},{i+1}) = {x}{y}" + ("  <- 'ba' => False" if x + y == "ba" else ""))
        if x + y == "ba":
            return False
    return True

print("kết quả:", trace_pairs("aabbab"))

  cặp (0,1) = aa
  cặp (1,2) = ab
  cặp (2,3) = bb
  cặp (3,4) = ba  <- 'ba' => False
kết quả: False


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Thử mọi cặp (baseline)

In [17]:
class Solution:
    def checkString(self, s: str) -> bool:
        n = len(s)
        for i in range(n):
            for j in range(i + 1, n):
                if s[i] == "b" and s[j] == "a":
                    return False
        return True

print([Solution().checkString(s) for s, _ in examples])

[True, False, True]


### Cách B — `a` cuối cùng đứng trước `b` đầu tiên

Cần xử lý `-1` (không có ký tự): nếu không có `b` hoặc không có `a` thì `True`.

In [18]:
class Solution:
    def checkString(self, s: str) -> bool:
        first_b = s.find("b")
        last_a = s.rfind("a")
        return first_b == -1 or last_a < first_b

print([Solution().checkString(s) for s, _ in examples])

[True, False, True]


Kiểm tra trường hợp không có `a`: `last_a = -1 < first_b` (vì `first_b ≥ 0`) → `True`. Không có `b`: `first_b == -1` → `True`. Gọn!

### Cách C — Một lượt với cờ (lời giải chính)

In [19]:
class Solution:
    def checkString(self, s: str) -> bool:
        seen_b = False
        for ch in s:
            if ch == "b":
                seen_b = True
            elif seen_b:
                return False
        return True

print([Solution().checkString(s) for s, _ in examples])

[True, False, True]


### Cách D — Không có chuỗi con `"ba"` (một dòng)

In [20]:
class Solution:
    def checkString(self, s: str) -> bool:
        return "ba" not in s

print([Solution().checkString(s) for s, _ in examples])

[True, False, True]


### Cách E — Chuỗi đã "được sắp xếp" chưa?

Dạng `aaa…bbb` chính là chuỗi **tăng dần** (vì `'a' < 'b'`). Kiểm tra bằng `sorted` (`O(n log n)`) hoặc bằng cặp kề `all(x <= y ...)` (`O(n)`).

In [21]:
class Solution:
    def checkString(self, s: str) -> bool:
        return all(x <= y for x, y in zip(s, s[1:]))

print([Solution().checkString(s) for s, _ in examples])
print("Cách sorted:", ["".join(sorted(s)) == s for s, _ in examples])

[True, False, True]
Cách sorted: [True, False, True]


## 9. Vì sao thuật toán đúng?

**Cách C — invariant:** tại đầu mỗi vòng lặp ở vị trí `i`, `seen_b == True` ⇔ có ít nhất một `b` trong `s[0:i]`; và chưa có vi phạm nào trong `s[0:i]` (nếu có, hàm đã `return False`).

- Gặp `a` khi `seen_b` ⇒ tồn tại `b` ở trước `a` ⇒ vi phạm thật ⇒ `False` đúng.
- Hết chuỗi mà không `return False` ⇒ không có `a` nào đứng sau một `b` ⇒ mọi `a` đứng trước mọi `b` ⇒ `True`.

**Cách D:** đã chứng minh ở mục 5: tồn tại cặp `b…a` (bất kỳ khoảng cách) ⇔ tồn tại cặp kề `"ba"`. (⇐ hiển nhiên; ⇒ xét lần đổi ký tự đầu tiên từ `b` sang `a`.)

**Cách B:** vi phạm ⇔ tồn tại `b` ở `i`, `a` ở `j`, `i < j` ⇔ `first_b < last_a` (chọn `i` nhỏ nhất, `j` lớn nhất).

## 10. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. Mọi cặp | `O(n²)` | `O(1)` | đúng nghĩa đen |
| B. `find` / `rfind` | `O(n)` | `O(1)` | hai lượt quét |
| C. Cờ một lượt | `O(n)` | `O(1)` | dừng sớm ở vi phạm đầu tiên |
| D. `"ba" not in s` | `O(n)` | `O(1)` | ngắn nhất, chạy bằng C |
| E. Cặp kề `<=` | `O(n)` | `O(1)` | (bản `sorted` là `O(n log n)`, `O(n)` bộ nhớ) |

## 11. Kiểm thử: mẫu đề, biên, **vét cạn mọi chuỗi ngắn**

Mỗi ký tự có 2 lựa chọn ⇒ có `2^n` chuỗi độ dài `n`. Với `n ≤ 12` ta thử **tất cả** (~8000 chuỗi) bằng `itertools.product`.

`product("ab", repeat=3)` sinh mọi bộ 3 ký tự từ `"ab"`: `('a','a','a'), ('a','a','b'), …`.

In [22]:
from itertools import product
print(["".join(p) for p in product("ab", repeat=3)])

['aaa', 'aab', 'aba', 'abb', 'baa', 'bab', 'bba', 'bbb']


In [23]:
def sol_A(s): return check_string_pairs(s)
def sol_B(s):
    fb = s.find("b"); return fb == -1 or s.rfind("a") < fb
def sol_C(s): return check_string(s)
def sol_D(s): return "ba" not in s
def sol_E(s): return all(x <= y for x, y in zip(s, s[1:]))
def sol_sorted(s): return "".join(sorted(s)) == s

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E, sol_sorted]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS ({actual})")

tests = examples + [("a", True), ("b", True), ("ab", True), ("ba", False), ("aaa", True), ("abba", False), ("bbbba", False)]
for s, exp in tests:
    for f in SOLS:
        check(f"{s!r} {f.__name__}", f(s), exp)

'aaabbb' sol_A             PASS (True)
'aaabbb' sol_B             PASS (True)
'aaabbb' sol_C             PASS (True)
'aaabbb' sol_D             PASS (True)
'aaabbb' sol_E             PASS (True)
'aaabbb' sol_sorted        PASS (True)
'abab' sol_A               PASS (False)
'abab' sol_B               PASS (False)
'abab' sol_C               PASS (False)
'abab' sol_D               PASS (False)
'abab' sol_E               PASS (False)
'abab' sol_sorted          PASS (False)
'bbb' sol_A                PASS (True)
'bbb' sol_B                PASS (True)
'bbb' sol_C                PASS (True)
'bbb' sol_D                PASS (True)
'bbb' sol_E                PASS (True)
'bbb' sol_sorted           PASS (True)
'a' sol_A                  PASS (True)
'a' sol_B                  PASS (True)
'a' sol_C                  PASS (True)
'a' sol_D                  PASS (True)
'a' sol_E                  PASS (True)
'a' sol_sorted             PASS (True)
'b' sol_A                  PASS (True)
'b' sol_B          

In [24]:
total = 0
for n in range(1, 13):
    for p in product("ab", repeat=n):
        s = "".join(p)
        outs = {f(s) for f in SOLS}
        assert len(outs) == 1, s
        # Định nghĩa gốc: s có dạng 'a'*k + 'b'*(n-k)
        assert outs.pop() == any(s == "a" * k + "b" * (n - k) for k in range(n + 1))
        total += 1
print("Vét cạn", total, "chuỗi độ dài 1..12: mọi cách khớp với định nghĩa")

Vét cạn 8190 chuỗi độ dài 1..12: mọi cách khớp với định nghĩa


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: So `find("a") < find("b")` (a **đầu tiên** trước b đầu tiên)

In [25]:
def wrong_first_first(s):
    return s.find("a") < s.find("b")        # SAI

for s in ["abab", "bbb"]:
    print(f"{s!r}: Sai -> {wrong_first_first(s)} | Đúng -> {check_string(s)}")

'abab': Sai -> True | Đúng -> False
'bbb': Sai -> True | Đúng -> True


**Vì sao sai:** (1) `"abab"`: a đầu (0) trước b đầu (1) nhưng vẫn có a đứng sau b. Cần so **a cuối** với **b đầu**. (2) `"bbb"`: `find("a") = -1 < 0` cho kết quả đúng một cách **tình cờ**; còn `"aaa"` cho `0 < -1` → `False` sai!

In [26]:
print("'aaa': Sai ->", wrong_first_first("aaa"), "| Đúng ->", check_string("aaa"))

'aaa': Sai -> False | Đúng -> True


### Lỗi 2: Quên xử lý `-1` khi không có `b`

In [27]:
def wrong_minus_one(s):
    return s.rfind("a") < s.find("b")       # SAI khi không có b

print("'aaa': Sai ->", wrong_minus_one("aaa"), "| Đúng ->", sol_B("aaa"))

'aaa': Sai -> False | Đúng -> True


### Lỗi 3: Kiểm tra `"ab" in s` (ngược chiều)

In [28]:
def wrong_ab(s):
    return "ab" in s                         # SAI: "aaabbb" có 'ab' nhưng "bbb" không

for s, exp in examples:
    print(f"{s!r}: Sai -> {wrong_ab(s)} | Đúng -> {exp}")

'aaabbb': Sai -> True | Đúng -> True
'abab': Sai -> True | Đúng -> False
'bbb': Sai -> False | Đúng -> True


### Lỗi 4: Tắt cờ lại khi gặp `a`

In [29]:
def wrong_reset(s):
    seen_b = False
    ok = True
    for ch in s:
        if ch == "b":
            seen_b = True
        else:
            ok = not seen_b          # SAI: ghi đè kết luận mỗi lần gặp a
            seen_b = False           # SAI: cờ phải "dính", không được tắt
    return ok

print("'abaab': Sai ->", wrong_reset("abaab"), "| Đúng ->", check_string("abaab"))

'abaab': Sai -> True | Đúng -> False


**Vì sao sai:** ở `"abaab"`, chữ `a` thứ hai (chỉ số 2) đã vi phạm, nhưng cờ bị tắt nên chữ `a` tiếp theo ghi đè `ok = True`. **Bài học:** cờ "đã từng gặp" chỉ bật, **không** tắt; và vi phạm phải được chốt (`return False`) **ngay khi** phát hiện.

## 13. Mẹo nhận dạng pattern

- "**Mọi** X trước **mọi** Y" ⇔ "**không có** Y nào trước X" ⇔ "**X cuối** < **Y đầu**".
- Chuỗi nhị phân "đã sắp xếp" ⇔ không có cặp kề giảm (`"ba"`, `"10"`).
- "Đã từng xảy ra chưa" → **biến cờ bool**.
- Miền nhỏ (chuỗi nhị phân ngắn) → **vét cạn bằng `itertools.product`** để kiểm chứng.

Bài liên quan: 1784 Check if Binary String Has at Most One Segment of Ones, 896 Monotonic Array, 1653 Minimum Deletions to Make String Balanced.

## 14. Biến thể

### 14.1. Mọi `b` đứng trước mọi `a` (đảo vai trò)

In [30]:
def all_b_before_a(s):
    return "ab" not in s

print([all_b_before_a(s) for s in ["bbaa", "abab", "aaa"]])

[True, False, True]


### 14.2. Chuỗi nhị phân có **tối đa một** đoạn số 1 liên tiếp (LeetCode 1784, chuỗi bắt đầu bằng '1')

In [31]:
def check_ones_segment(s):
    return "01" not in s

print(check_ones_segment("1001"), check_ones_segment("110"))

False True


### 14.3. Số ký tự **tối thiểu cần xóa** để thành dạng `a…ab…b` (LeetCode 1653)

Duyệt trái→phải, giữ `b_count` (số `b` đã thấy) và `dels` (số xóa tối thiểu cho tiền tố). Gặp `a`: hoặc xóa `a` này (`dels + 1`), hoặc xóa mọi `b` trước nó (`b_count`).

In [32]:
def min_deletions(s):
    b_count = dels = 0
    for ch in s:
        if ch == "b":
            b_count += 1
        else:
            dels = min(dels + 1, b_count)
    return dels

print(min_deletions("aababbab"), min_deletions("bbaaaaabb"), min_deletions("aaabbb"))

2 2 0


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Vị trí `a` cuối cùng (không dùng `rfind`), không có trả `-1`

In [33]:
def last_a(s):
    # TODO: viết code ở đây
    pass

# assert last_a("abab") == 2 and last_a("bbb") == -1

### Bài 2 — Tự viết lại lời giải #2124 bằng cờ

In [34]:
def check_string_ex(s):
    # TODO: viết code ở đây
    pass

# assert check_string_ex("aaabbb") and not check_string_ex("abab") and check_string_ex("bbb")

### Bài 3 — Đếm số cặp vi phạm `(i, j)`, `i < j`, `s[i]='b'`, `s[j]='a'` trong `O(n)`

In [35]:
def count_violations(s):
    # TODO: viết code ở đây
    pass

# assert count_violations("abab") == 1 and count_violations("bbaa") == 4 and count_violations("aabb") == 0

### Bài 4 — Biến thể: chuỗi gồm `a`, `b`, `c`; kiểm tra dạng `a…ab…bc…c`

In [36]:
def check_abc(s):
    # TODO: viết code ở đây
    pass

# assert check_abc("aabbcc") and check_abc("acc") and not check_abc("abcb") and not check_abc("ca")

### Bài 5 — Tự tạo test làm lộ **Lỗi 1** (`find("a") < find("b")`)

In [37]:
my_tests = [
    # TODO: các chuỗi s mà wrong_first_first(s) != check_string(s)
]
# for s in my_tests:
#     assert wrong_first_first(s) != check_string(s)

### Bài 6 — Mini challenge: tính `min_deletions(s)` bằng cách thử mọi điểm cắt (`O(n)` với tiền tố)
Ở điểm cắt `k`: xóa mọi `b` trong `s[:k]` và mọi `a` trong `s[k:]`. Lấy min theo `k`.

In [38]:
def min_deletions_cut(s):
    # TODO: viết code ở đây
    pass

# assert min_deletions_cut("aababbab") == 2 and min_deletions_cut("bbaaaaabb") == 2

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | duyệt và ghi nhớ | cập nhật khi gặp `a` | khởi tạo `-1` |
| 2 | cờ `seen_b` | gặp `a` khi cờ bật | `return False` ngay |
| 3 | mỗi `a` vi phạm với mọi `b` trước nó | đếm `b` đã thấy | gặp `a`: cộng `b_count` |
| 4 | chuỗi phải tăng dần | không có cặp kề giảm | `all(x <= y for x, y in zip(s, s[1:]))` |
| 5 | Lỗi 1 chỉ so a đầu | cần a đầu < b đầu nhưng có a sau b | `"abab"`, `"aaa"` |
| 6 | mảng tiền tố số `b` | số `a` hậu tố = tổng `a` − `a` tiền tố | `min` trên `k = 0..n` |

## 17. Đáp án tham khảo

In [39]:
def last_a(s):
    pos = -1
    for i, ch in enumerate(s):
        if ch == "a":
            pos = i
    return pos

def check_string_ex(s):
    seen_b = False
    for ch in s:
        if ch == "b":
            seen_b = True
        elif seen_b:
            return False
    return True

def count_violations(s):
    b_count = total = 0
    for ch in s:
        if ch == "b":
            b_count += 1
        else:
            total += b_count
    return total

def check_abc(s):
    return all(x <= y for x, y in zip(s, s[1:]))

my_tests = ["abab", "aaa", "aba"]

def min_deletions_cut(s):
    total_a = s.count("a")
    best = total_a                      # k = 0: xóa mọi a
    b_prefix = a_prefix = 0
    for ch in s:
        if ch == "a":
            a_prefix += 1
        else:
            b_prefix += 1
        best = min(best, b_prefix + (total_a - a_prefix))
    return best

assert last_a("abab") == 2 and last_a("bbb") == -1
assert check_string_ex("aaabbb") and not check_string_ex("abab") and check_string_ex("bbb")
assert count_violations("abab") == 1 and count_violations("bbaa") == 4 and count_violations("aabb") == 0
assert check_abc("aabbcc") and check_abc("acc") and not check_abc("abcb") and not check_abc("ca")
for s in my_tests:
    assert wrong_first_first(s) != check_string(s)
assert min_deletions_cut("aababbab") == 2 and min_deletions_cut("bbaaaaabb") == 2
for p in product("ab", repeat=8):
    s = "".join(p)
    assert min_deletions_cut(s) == min_deletions(s)
print("Tất cả đáp án PASS (bài 6 đã đối chiếu vét cạn với min_deletions)")

Tất cả đáp án PASS (bài 6 đã đối chiếu vét cạn với min_deletions)


**Giải thích bằng lời:**
- Bài 1: ghi đè vị trí mỗi khi gặp `a` → cuối cùng là vị trí `a` cuối.
- Bài 2: lời giải chính.
- Bài 3: mỗi `a` tạo vi phạm với **mọi** `b` đứng trước nó → cộng dồn `b_count`. Bài gốc ⇔ `count_violations(s) == 0`.
- Bài 4: tổng quát hóa "đã sắp xếp" cho 3 ký tự.
- Bài 5: Lỗi 1 sai khi có `a` sau `b` mà `a` đầu vẫn đứng trước `b` đầu, hoặc khi không có `b`.
- Bài 6: thử mọi ranh giới giữa khối `a` và khối `b`; tiền tố giúp mỗi ranh giới tính `O(1)`.

## 18. Tổng kết — template + checklist

**Template:**
```python
return "ba" not in s                  # ngắn nhất
# hoặc cờ một lượt:
seen_b = False
for ch in s:
    if ch == "b": seen_b = True
    elif seen_b: return False
return True
```

**Checklist trước khi submit:**
- [ ] Chuỗi không có `a` hoặc không có `b` → `True`.
- [ ] So **a cuối** với **b đầu**, không phải a đầu.
- [ ] Xử lý `-1` của `find/rfind`.
- [ ] Cờ chỉ bật, không reset.
- [ ] Đúng tên method `checkString`.